# 08 - A Second Leakage Bug, and Turning This Into Something Usable

Two things in this notebook:

1. **A genuine temporal leakage bug in the historical-rate features**,
   found by asking a simple question that hadn't been asked yet: *at
   the moment order X was placed, could its seller's "history" have
   actually been observed?* It turns out the answer was often no.
2. Everything needed to actually **use** the model on a new order
   without writing throwaway code each time: a rate lookup so a caller
   only needs a `seller_id`, not three hand-computed features; an
   integrity-checked model loader; a batch scorer; and a command-line
   tool (`score.py`) wrapping all of it.

In [1]:
import sys, os, json
sys.path.append("../src")
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import roc_auc_score, average_precision_score
import joblib

from features import (FEATURE_COLUMNS, TARGET_COLUMN, add_time_features,
                      add_historical_rates, add_historical_rates_by_purchase_order,
                      build_feature_table)
from modeling import stratified_split

## 1. The bug

The historical seller/category/region delay rate was built (notebooks
04-07) by averaging the outcomes of every earlier-**purchased** order
in the group. That was already careful to sort by time and exclude the
order itself - but "purchased earlier" is not the same thing as
"outcome known yet". A typical order takes about ten days to arrive:

`order bought 3 days ago -> not delivered yet -> is_delayed doesn't exist yet`

so a meaningful slice of what the old feature called "history" was
actually unobservable at the time. The fallback made it worse: a group
with no *purchased* history yet fell back to the delay rate computed
over **the entire dataset, including orders that hadn't happened yet.**

In [2]:
raw = pd.read_csv("../data/processed/ontime_joined_raw.csv", parse_dates=[
    "order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
    "order_delivered_customer_date", "order_estimated_delivery_date",
])
raw["days_to_deliver"] = (raw.order_delivered_customer_date - raw.order_purchase_timestamp).dt.total_seconds() / 86400
print(f"Median time from purchase to delivery: {raw.days_to_deliver.median():.1f} days "
      f"(75th percentile: {raw.days_to_deliver.quantile(.75):.1f} days)")

Median time from purchase to delivery: 10.2 days (75th percentile: 15.7 days)


How much of the old feature's "history" was genuinely unobservable?
For each state, compare how many earlier-purchased orders existed
against how many were actually delivered yet:

In [3]:
r = raw.sort_values("order_purchase_timestamp").reset_index(drop=True)
frac_unknown = []
for st, g in r.groupby("customer_state"):
    if len(g) < 500:
        continue
    p, d = g.order_purchase_timestamp.values, np.sort(g.order_delivered_customer_date.values)
    n_before = np.arange(len(g))
    n_known = np.searchsorted(d, p, side="left")
    m = n_before > 0
    frac_unknown.append(1 - n_known[m] / n_before[m])
fu = np.concatenate(frac_unknown)
print(f"Share of the old feature's 'history' whose outcome was not yet observable: "
      f"mean {fu.mean():.1%}, median {np.median(fu):.1%}")

Share of the old feature's 'history' whose outcome was not yet observable: mean 7.8%, median 5.7%


## 2. The fix

`add_historical_rates()` in `src/features.py` was rewritten so an
order's history is same-group orders **delivered strictly before** its
own purchase time - checked directly against `order_delivered_customer_date`,
not inferred from purchase order. The fallback is the platform-wide
rate similarly restricted to what was deliverable by that moment, and
only orders placed before anything at all had been delivered get a
hard 0.0. None of this ever looks at the future. The old function is
kept in the codebase, renamed to make what it does explicit
(`add_historical_rates_by_purchase_order`), purely so the two can be
compared here rather than the old behaviour just disappearing.

`features.py` also gained a second, smaller guard while this was being
worked on: `build_feature_table()` now checks that every column it
needs is actually present and names the missing ones, instead of
failing partway through with a `KeyError` that doesn't say why.

In [4]:
try:
    build_feature_table(pd.DataFrame({"a": [1]}))
except ValueError as e:
    print("Deliberately incomplete input ->", str(e)[:140], "...")

Deliberately incomplete input -> build_feature_table: the joined table is missing 30 required column(s): ['avg_description_length', 'avg_freight', 'avg_item_price', 'avg_nam ...


## 3. Does the fix change model performance?

Both feature versions, built from the identical joined table, fed to
the identical tuned XGBoost architecture, on the identical train/test
split.

In [5]:
new_feat = build_feature_table(raw)

old_feat = add_time_features(raw)
for key, col in [("primary_seller_id", "seller_historical_delay_rate"),
                 ("product_category", "category_historical_delay_rate"),
                 ("customer_state", "customer_state_historical_delay_rate")]:
    old_feat = add_historical_rates_by_purchase_order(old_feat, key, col)
for c in ["total_weight_g", "avg_weight_g", "total_volume_cm3", "seller_customer_distance_km",
          "total_payment_value", "n_payment_installments", "n_payment_methods",
          "avg_photos_qty", "avg_name_length", "avg_description_length"]:
    old_feat[c] = old_feat[c].fillna(old_feat[c].median())
old_feat["product_category"] = old_feat["product_category"].fillna("unknown")
old_feat["payment_type"] = old_feat["payment_type"].fillna("unknown")

ready = pd.read_csv("../data/processed/ontime_model_ready.csv", usecols=["order_id", TARGET_COLUMN])
tr_ids, te_ids = [set(d.order_id) for d in stratified_split(ready)]
final_model = joblib.load("../models/ontime_final_model.joblib")
arch = clone(final_model)

results = {}
for label, tbl in [("OLD (purchase-order history)", old_feat), ("NEW (delivery-order history)", new_feat)]:
    tr, te = tbl[tbl.order_id.isin(tr_ids)], tbl[tbl.order_id.isin(te_ids)]
    m = clone(arch)
    m.fit(tr[FEATURE_COLUMNS], tr[TARGET_COLUMN])
    p = m.predict_proba(te[FEATURE_COLUMNS])[:, 1]
    results[label] = {"roc_auc": roc_auc_score(te[TARGET_COLUMN], p), "pr_auc": average_precision_score(te[TARGET_COLUMN], p)}
    print(f"{label:32s} ROC-AUC={results[label]['roc_auc']:.3f}  PR-AUC={results[label]['pr_auc']:.3f}")

j = old_feat[["order_id", "seller_historical_delay_rate"]].merge(
    new_feat[["order_id", "seller_historical_delay_rate"]], on="order_id", suffixes=("_old", "_new"))
print(f"\ncorrelation between old and new seller-rate values: "
      f"{np.corrcoef(j.seller_historical_delay_rate_old, j.seller_historical_delay_rate_new)[0,1]:.3f}  "
      f"(the two features disagree substantially at the individual-order level)")

OLD (purchase-order history)     ROC-AUC=0.801  PR-AUC=0.340


NEW (delivery-order history)     ROC-AUC=0.805  PR-AUC=0.340

correlation between old and new seller-rate values: 0.683  (the two features disagree substantially at the individual-order level)


The two feature versions are meaningfully different for individual
orders (correlation well below 1.0) but land at almost the same
aggregate ROC-AUC. **That is worth being honest about both ways**: the
leak was real and the fix is correct regardless of the aggregate
metric, but on this particular dataset the practical performance cost
of the bug happened to be small. The next dataset might not be so
forgiving, which is the actual argument for fixing it rather than
shrugging at a metric that looks fine.

## 4. Production artifacts

Three files, saved alongside the model, that a caller needs to score a
genuinely new order without recomputing anything from the training set:

In [6]:
from artifacts import build_rate_lookup, save_json, save_model_metadata, load_model_checked
from inference_validation import build_reference_stats, save_reference_stats, load_reference_stats
from modeling import NUMERIC, CATEGORICAL

joined = pd.read_csv("../data/processed/ontime_joined_raw.csv",
                     usecols=["order_id", "primary_seller_id", "product_category", "customer_state", "is_delayed"])
lookup = build_rate_lookup(joined)
save_json(lookup, "../models/rate_lookup.json")
print(f"rate_lookup.json: {len(lookup['seller']):,} sellers, {len(lookup['category'])} categories, "
      f"{len(lookup['customer_state'])} states, global rate {lookup['global_rate']:.3f}")

df = pd.read_csv("../data/processed/ontime_model_ready.csv", parse_dates=["order_purchase_timestamp"])
train_df, test_df = stratified_split(df)
RATE_COLS = ["seller_historical_delay_rate", "category_historical_delay_rate", "customer_state_historical_delay_rate"]
stats = build_reference_stats(train_df, NUMERIC, CATEGORICAL, rate_cols=RATE_COLS)
save_reference_stats(stats, "../models/reference_stats.json")

metrics = json.load(open("../dashboard/model_metrics.json"))
meta = save_model_metadata("../models/ontime_final_model.joblib", FEATURE_COLUMNS, NUMERIC, CATEGORICAL,
                           metrics["selected_threshold"], metrics["xgboost_tuned"], "../models/model_metadata.json")
print(f"model_metadata.json: threshold={meta['threshold']:.3f}, sha256={meta['model_sha256'][:16]}...")
print("library_versions:", meta["library_versions"])

rate_lookup.json: 2,959 sellers, 74 categories, 27 states, global rate 0.081


model_metadata.json: threshold=0.618, sha256=28e2f3a313f51cbc...
library_versions: {'scikit-learn': '1.8.0', 'xgboost': '3.4.1', 'pandas': '3.0.2', 'numpy': '2.4.4'}


`load_model_checked()` refuses to load the model if the file on disk
doesn't match the hash recorded when it was saved, or if the current
code's feature list no longer matches what the model was trained on -
both are silent-wrong-answer risks for a plain `joblib.load`, not
loud-failure risks.

In [7]:
from artifacts import ModelIntegrityError
model, meta = load_model_checked("../models/ontime_final_model.joblib", "../models/model_metadata.json", FEATURE_COLUMNS)
print("Loaded and verified OK.")
try:
    load_model_checked("../models/ontime_final_model.joblib", "../models/model_metadata.json", FEATURE_COLUMNS[:-1])
except ModelIntegrityError as e:
    print("Deliberately wrong feature list ->", str(e)[:120], "...")

Loaded and verified OK.
Deliberately wrong feature list -> Feature list in the current code differs from the one this model was trained on (missing from code: ['customer_state_his ...


## 5. Scoring new orders in batch

`src/scoring.py` fills the three rate features from the lookup (using
`seller_id`, `product_category`, `customer_state`), validates every
row, scores the valid ones, and attaches reasons to anything flagged -
in one call, for a whole table at once.

In [8]:
from scoring import score_orders, required_input_columns, summarise

sample = test_df.sample(500, random_state=2).merge(
    joined[["order_id", "primary_seller_id"]], on="order_id", how="left"
).rename(columns={"primary_seller_id": "seller_id"})
new_orders = sample.drop(columns=RATE_COLS)  # a caller would never have these

scored = score_orders(new_orders, model, meta["threshold"], stats, lookup)
print(summarise(scored))
scored.loc[scored.flagged == True, ["order_id", "probability", "risk_level", "reasons"]].head(5)

Rows: 500   scored: 500   rejected: 0
Flagged as at-risk: 38 of 500
Risk levels: LOW=350, MEDIUM=106, HIGH=30, CRITICAL=14
Low-confidence rows: 0


,order_id,probability,risk_level,reasons
7,a67c02dc9c53a5bae76418bff0a534cf,0.7475,HIGH,Seasonal timing of the order (+0.79); avg frei...
12,28f004902453ef50891cc5871ff07c2b,0.8572,CRITICAL,Seasonal timing of the order (+1.01); Delivery...
20,2edcf6adbf3719366560d6863cb28799,0.7966,HIGH,Length of promised delivery window (+0.77); Se...
46,2c53bd1388754e218f18b97bf025d935,0.6188,HIGH,Long shipping distance (+0.66); Length of prom...
71,b951bb66b451384026a3308f84f1eab9,0.9313,CRITICAL,Seasonal timing of the order (+0.95); Length o...


## 6. The command line

Everything above is also wrapped in `score.py` at the project root, so
scoring a CSV of new orders doesn't require writing any Python:

```
python score.py --template            # writes an example input file
python score.py --input orders.csv    # scores it, writes scored_orders.csv
```

Every input row produces exactly one output row - `SCORED` or
`REJECTED`, never silently dropped.

## 7. Automated tests

The scratch checks used throughout notebooks 07-08 (does the validator
reject bad data without crashing, does an order's own outcome ever
leak into its own feature, does the guardrail warn on the right
things) are now a real test suite under `tests/`, runnable with
`pytest tests/` from the project root - 62 tests, covering the raw-data
validator, the historical-rate timing properties, the inference
guardrails, and the saved model/scoring pipeline end to end. Each of
the bugs found across notebooks 07-08 has a corresponding test that
fails if that bug is reintroduced - checked directly, by temporarily
undoing each fix and confirming the relevant test goes red before
putting the fix back.